# 00 · Preparación de datos
**Iván Ramiro Pinzón Pinto** — Ciencia de Datos, Universidad Externado de Colombia · Tutora: Yessica Velásquez

*Implementa §3.1–3.3 (carga, estandarización de estaciones y consolidación de validaciones).*

**Se corre una sola vez.** Consolida los 24 archivos mensuales y guarda los insumos limpios en `data/intermedia/`. Los demás cuadernos cargan esos `.parquet` y no vuelven a leer los Excel.


In [17]:
import pandas as pd
import numpy as np
import json
import re
import os
import warnings
from glob import glob
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, silhouette_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from itertools import combinations
from matplotlib.patches import Patch
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.inspection import permutation_importance
from sklearn.metrics import adjusted_rand_score, silhouette_samples
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 1. Configuración y carga de datos

**Instrucción:** Ajustar `DATA_DIR` a la carpeta donde están los 24 archivos de validaciones, y las rutas de los archivos de soporte a su ubicación local.

In [18]:
# === Raíz del repo: sirve corras desde la raíz o desde notebooks/ ===
RAIZ = os.getcwd()
if os.path.basename(RAIZ) == 'notebooks':
    RAIZ = os.path.dirname(RAIZ)

DATA_RAW = os.path.join(RAIZ, 'data', 'raw')         # los 24 .xlsx
DATA_SOP = os.path.join(RAIZ, 'data', 'soporte')     # archivos de soporte
DATA_INT = os.path.join(RAIZ, 'data', 'intermedia')  # .parquet de salida
os.makedirs(DATA_INT, exist_ok=True)

# Archivos de soporte
MATRIZ_PATH             = os.path.join(DATA_SOP, 'matriz_paradas_troncales_dic2025.csv')
SERVICIOS_PATH          = os.path.join(DATA_SOP, 'Servicios__Rutas_Troncales_y_Zonales_.csv')
ESTACIONES_OFICIAL_PATH = os.path.join(DATA_SOP, 'transmilenio_troncal_estaciones.csv')  # <- ver nota
CODE_MAPPING_PATH       = os.path.join(DATA_SOP, 'code_to_matriz.json')

# Verificar
for path in [MATRIZ_PATH, SERVICIOS_PATH, ESTACIONES_OFICIAL_PATH, CODE_MAPPING_PATH]:
    status = "OK" if os.path.exists(path) else "FALTA"
    print(f"  {status}: {os.path.basename(path)}")

# Validaciones
val_files = sorted(glob(os.path.join(DATA_RAW, '*Validaciones*Troncales*.xlsx')))
print(f"\nArchivos de validaciones encontrados: {len(val_files)}")
for f in val_files:
    print(f"  {os.path.basename(f)}")

  OK: matriz_paradas_troncales_dic2025.csv
  OK: Servicios__Rutas_Troncales_y_Zonales_.csv
  OK: transmilenio_troncal_estaciones.csv
  OK: code_to_matriz.json

Archivos de validaciones encontrados: 24
  01 TM Resumen de Validaciones Troncales al 31 Ene 2024 Intervalo 15 Mint.xlsx
  01 TM Resumen de Validaciones Troncales al 31 de Enero 2025 Intervalo 15 Mint.xlsx
  02 TM Resumen de Validaciones Troncales al 28 de Febrero 2025 Intervalo 15 Mint.xlsx
  02 TM Resumen de Validaciones Troncales al 29 Feb 2024 Intervalo 15 Mint.xlsx
  03 TM Resumen de Validaciones Troncales al 31 Mar 2024 Intervalo 15 Mint.xlsx
  03 TM Resumen de Validaciones Troncales al 31 de Marzo 2025 Intervalo 15 Mint.xlsx
  04 TM Resumen de Validaciones Troncales al 30 Abr 2024 Intervalo 15 Mint.xlsx
  04 TM Resumen de Validaciones Troncales al 30 de Abril 2025 Intervalo 15 Mint.xlsx
  05 TM Resumen de Validaciones Troncales al 31 May 2024 Intervalo 15 Mint.xlsx
  05 TM Resumen de Validaciones Troncales al 31 de Mayo 2

In [19]:
# =============================================
# CARGA DE DATOS DE SOPORTE
# =============================================

# 1. Matriz ruta-estación
matriz = pd.read_csv(MATRIZ_PATH)
print(f"Matriz ruta-estación: {len(matriz)} filas, "
      f"{matriz['ruta'].nunique()} rutas, {matriz['estacion'].nunique()} estaciones")

# 2. Servicios troncales (para tipo de bus)
servicios = pd.read_csv(SERVICIOS_PATH)
troncal_serv = servicios[servicios['tip_operac'] == 1][['cod_ruta', 'tip_bus']].drop_duplicates()
print(f"Servicios troncales: {len(troncal_serv)} rutas con tipo de bus")

# 3. Mapeo de códigos de estación
with open(CODE_MAPPING_PATH, 'r', encoding='utf-8') as f:
    code_to_matriz = json.load(f)
print(f"Mapeo de códigos: {len(code_to_matriz)} códigos -> "
      f"{len(set(code_to_matriz.values()))} estaciones únicas")

Matriz ruta-estación: 1547 filas, 101 rutas, 232 estaciones
Servicios troncales: 111 rutas con tipo de bus
Mapeo de códigos: 155 códigos -> 148 estaciones únicas


## 2. Estandarización de estaciones (mapeo por código)

Las validaciones TuLlave usan el formato `(XXXXX) Nombre Estación`. El código numérico es el identificador estable; el nombre textual varía entre meses.

In [20]:
def extraer_codigo(estacion_raw):
    """Extrae el código numérico del nombre crudo de estación."""
    m = re.match(r'\((\d+)\)', str(estacion_raw).strip())
    return m.group(1) if m else None

def mapear_estacion(estacion_raw):
    """Mapea nombre crudo de validaciones al nombre estandarizado de la matriz."""
    code = extraer_codigo(estacion_raw)
    if code and code in code_to_matriz:
        return code_to_matriz[code]
    return None

# Verificación
tests = [
    '(02000) Cabecera Autopista Norte',
    '(05000) Portal Américas',
    '(09110) Avenida Jimenez',
    '(06101) El Tiempo',
    '(06101) El Tiempo - Camara de Comercio de Bogota',
]
print("Verificación del mapeo por código:")
for t in tests:
    print(f"  {t:55s} -> {mapear_estacion(t)}")

Verificación del mapeo por código:
  (02000) Cabecera Autopista Norte                        -> Portal Norte
  (05000) Portal Américas                                 -> Portal Américas
  (09110) Avenida Jimenez                                 -> AV. Jiménez
  (06101) El Tiempo                                       -> El Tiempo - CCB
  (06101) El Tiempo - Camara de Comercio de Bogota        -> El Tiempo - CCB


## 3. Consolidación de validaciones (24 meses)

Se cargan los 24 archivos mensuales, se transforman de formato ancho a largo, se aplica el mapeo por código y se agregan validaciones por estación-fecha-intervalo.

In [21]:
def leer_validaciones(filepath):
    """Lee un archivo mensual de validaciones y lo transforma a formato largo."""
    df = pd.read_excel(filepath, header=None, skiprows=6)
    df.columns = df.iloc[0]
    df = df.iloc[1:].copy()
    fname = os.path.basename(filepath)
    
    meta_cols = ['Fase', 'Línea', 'Estación', 'Acceso de Estación', 'Intervalo']
    day_cols = [c for c in df.columns if c not in meta_cols and c is not None]
    
    df = df[df['Estación'].notna()].copy()
    df['estacion'] = df['Estación'].apply(mapear_estacion)
    
    n_before = len(df)
    df = df[df['estacion'].notna()].copy()
    n_after = len(df)
    
    id_vars = ['estacion', 'Intervalo']
    value_vars = [c for c in day_cols if str(c).strip() not in ['', 'None', 'Total general']]
    
    melted = df.melt(id_vars=id_vars, value_vars=value_vars,
                     var_name='fecha', value_name='validaciones')
    
    melted['validaciones'] = pd.to_numeric(melted['validaciones'], errors='coerce').fillna(0).astype(int)
    melted['fecha'] = pd.to_datetime(melted['fecha'], errors='coerce')
    melted = melted.dropna(subset=['fecha'])
    melted = melted.rename(columns={'Intervalo': 'intervalo'})
    
    result = melted.groupby(['estacion', 'fecha', 'intervalo'])['validaciones'].sum().reset_index()
    print(f"  {fname}: {n_after}/{n_before} filas mapeadas -> {len(result)} registros")
    return result

# Cargar todos los archivos
print("Cargando validaciones...\n")
all_dfs = []
for f in val_files:
    try:
        df = leer_validaciones(f)
        all_dfs.append(df)
    except Exception as e:
        print(f"  ERROR en {os.path.basename(f)}: {e}")

validaciones = pd.concat(all_dfs, ignore_index=True)
validaciones = validaciones.groupby(['estacion', 'fecha', 'intervalo'])['validaciones'].sum().reset_index()

print(f"\n{'='*50}")
print(f"CONSOLIDADO FINAL")
print(f"  Filas totales:      {len(validaciones):,}")
print(f"  Estaciones únicas:  {validaciones['estacion'].nunique()}")
print(f"  Rango de fechas:    {validaciones['fecha'].min()} a {validaciones['fecha'].max()}")
print(f"  Meses cubiertos:    {validaciones['fecha'].dt.to_period('M').nunique()}")

Cargando validaciones...

  01 TM Resumen de Validaciones Troncales al 31 Ene 2024 Intervalo 15 Mint.xlsx: 37975/69821 filas mapeadas -> 358081 registros
  01 TM Resumen de Validaciones Troncales al 31 de Enero 2025 Intervalo 15 Mint.xlsx: 38446/52567 filas mapeadas -> 346363 registros
  02 TM Resumen de Validaciones Troncales al 28 de Febrero 2025 Intervalo 15 Mint.xlsx: 37638/56448 filas mapeadas -> 347975 registros
  02 TM Resumen de Validaciones Troncales al 29 Feb 2024 Intervalo 15 Mint.xlsx: 38162/52953 filas mapeadas -> 359507 registros
  03 TM Resumen de Validaciones Troncales al 31 Mar 2024 Intervalo 15 Mint.xlsx: 37627/53389 filas mapeadas -> 359383 registros
  03 TM Resumen de Validaciones Troncales al 31 de Marzo 2025 Intervalo 15 Mint.xlsx: 37584/52608 filas mapeadas -> 347944 registros
  04 TM Resumen de Validaciones Troncales al 30 Abr 2024 Intervalo 15 Mint.xlsx: 37682/51765 filas mapeadas -> 358515 registros
  04 TM Resumen de Validaciones Troncales al 30 de Abril 2025

In [22]:
# =============================================
# VARIABLES TEMPORALES
# =============================================

validaciones['hora'] = validaciones['intervalo'].apply(
    lambda x: int(str(x).split(':')[0]) if pd.notna(x) and ':' in str(x) else np.nan
)
validaciones['dia_semana'] = validaciones['fecha'].dt.dayofweek
validaciones['mes'] = validaciones['fecha'].dt.month
validaciones['anio'] = validaciones['fecha'].dt.year
validaciones['es_fin_semana'] = validaciones['dia_semana'].isin([5, 6]).astype(int)

# Festivos Colombia 2024-2025
festivos = pd.to_datetime([
    '2024-01-01', '2024-01-08', '2024-03-25', '2024-03-28', '2024-03-29',
    '2024-05-01', '2024-05-13', '2024-06-03', '2024-06-10', '2024-07-01',
    '2024-07-20', '2024-08-07', '2024-08-19', '2024-10-14', '2024-11-04',
    '2024-11-11', '2024-12-08', '2024-12-25',
    '2025-01-01', '2025-01-06', '2025-03-24', '2025-04-17', '2025-04-18',
    '2025-05-01', '2025-06-02', '2025-06-23', '2025-06-30', '2025-07-20',
    '2025-08-07', '2025-08-18', '2025-10-13', '2025-11-03', '2025-11-17',
    '2025-12-08', '2025-12-25',
])
validaciones['es_festivo'] = validaciones['fecha'].isin(festivos).astype(int)

print("Variables temporales agregadas.")
print(f"Distribución por día de la semana:")
print(validaciones.groupby('dia_semana')['validaciones'].mean().round(0))

Variables temporales agregadas.
Distribución por día de la semana:
dia_semana
0    137.0
1    160.0
2    156.0
3    154.0
4    158.0
5    110.0
6     53.0
Name: validaciones, dtype: float64


In [23]:
# =============================================
# CORRECCIÓN: Unificar nombres duplicados en la matriz
# =============================================

UNIFICAR_MATRIZ = {
    'AV 1° de Mayo': 'AV. 1° de Mayo',
    'AV. 1° Mayo': 'AV. 1° de Mayo',
    'Alcalá - Col S. Tomás Dominicos': 'Alcalá - Col. S. Tomás Dominicos',
    'Calle 40 Sur': 'Calle 40 S.',
    'Calle 40S.': 'Calle 40 S.',
    'Castellana': 'La Castellana',
    'Guatoque-Veraguas': 'Guatoque - Veraguas',
    'KR. 90': 'KR 90',
    'Las Aguas - C. Col. Americano': 'Las Aguas - C. Col Americano',
    'NQS - Calle 30 Sur': 'NQS - Calle 30 S.',
    'NQS - Calle 30S.': 'NQS - Calle 30 S.',
    'NQS - Calle 38A Sur': 'NQS - Calle 38A S.',
    'Portal Eldorado- C.C. NUESTRO BOGOTÁ': 'Portal Eldorado - C.C. NUESTRO BOGOTÁ',
    'Suba - TV. 91': 'Suba - TV 91',
    'Tygua-San José': 'Tygua - San José',
    'Terreros Hsp. C.V.': 'Terreros - Hospital C.V.',
}

antes = matriz['estacion'].nunique()
matriz['estacion'] = matriz['estacion'].replace(UNIFICAR_MATRIZ)
despues = matriz['estacion'].nunique()

print(f"Estaciones antes: {antes}")
print(f"Estaciones después: {despues}")
print(f"Duplicados unificados: {antes - despues}")

Estaciones antes: 232
Estaciones después: 216
Duplicados unificados: 16


In [24]:
# === GUARDA LOS INSUMOS LIMPIOS (costura hacia los demás cuadernos) ===
import os
from pathlib import Path

# raíz del proyecto: sube hasta encontrar un marcador del repo (sirva donde sirva el kernel)
def raiz_proyecto(marcadores=('.git', 'requirements.txt', 'README.md')):
    d = Path.cwd().resolve()
    for cand in (d, *d.parents):
        if any((cand / m).exists() for m in marcadores):
            return cand
    return d  # respaldo

RAIZ = raiz_proyecto()
DATA_INT = os.path.join(RAIZ, 'data', 'intermedia')
os.makedirs(DATA_INT, exist_ok=True)

validaciones.to_parquet(os.path.join(DATA_INT, 'validaciones_consolidadas.parquet'), index=False)
matriz.to_parquet(os.path.join(DATA_INT, 'matriz_estandarizada.parquet'), index=False)
troncal_serv.to_parquet(os.path.join(DATA_INT, 'troncal_serv.parquet'), index=False)
print(f'Guardados en: {DATA_INT}')   


Guardados en: C:\Users\ivanr\OneDrive\Desktop\Transmilenio Data\data\intermedia


In [25]:
# ============================================================
# COBERTURA: 150 del catálogo vs. validaciones — diagnóstico por estación
# Caza los desajustes de formato
# ============================================================
import os, json
import pandas as pd

RUTA_GEOJSON = os.path.join(DATA_SOP, 'catalogo_estaciones_troncales.geojson')
RUTA_MAPEO   = os.path.join(DATA_SOP, 'code_to_matriz.json')

def z5(c):
    """Normaliza un código a 5 dígitos: '9005' -> '09005'."""
    return str(c).strip().zfill(5)

# --- 1) catálogo oficial (150) ---
with open(RUTA_GEOJSON, encoding='utf-8') as f:
    gj = json.load(f)
cat = pd.DataFrame([{
    'codigo_raw'     : str(feat['properties'].get('num_est')),
    'codigo'         : z5(feat['properties'].get('num_est')),
    'nombre_catalogo': str(feat['properties'].get('nom_est', '')).strip(),
} for feat in gj['features']]).drop_duplicates('codigo')
print(f"Catálogo oficial: {len(cat)} estaciones")

# --- 2) códigos que SÍ tienen validaciones (del diccionario de mapeo) ---
with open(RUTA_MAPEO, encoding='utf-8') as f:
    code_to_matriz = json.load(f)
val_raw = {str(c).strip() for c in code_to_matriz}          # tal cual están las llaves
val_5   = {z5(c) for c in code_to_matriz}                   # normalizadas a 5 dígitos
nombres_map = {str(v).strip().lower() for v in code_to_matriz.values()}
print(f"Mapeo: {len(val_raw)} códigos -> {len(set(code_to_matriz.values()))} estaciones únicas")

# --- 3) diagnóstico estación por estación ---
def diagnosticar(r):
    if r['codigo'] in val_5:
        # cruza al normalizar: ¿habría fallado SIN zfill?
        return f"CUBIERTA (requería zfill: {r['codigo_raw']} -> {r['codigo']})" \
               if r['codigo_raw'] not in val_raw else "CUBIERTA"
    if r['nombre_catalogo'].lower() in nombres_map:
        return "REVISAR: nombre coincide pero el código no cruza"
    return "SIN VALIDACIONES (código ausente: cerrada / sin datos / nombre distinto)"

cat['estado'] = cat.apply(diagnosticar, axis=1)

# --- 4) resumen ---
print("\nResumen de cobertura:")
print(cat['estado'].str.split(' (', regex=False).str[0].value_counts().to_string())
cubiertas = cat['estado'].str.startswith('CUBIERTA').sum()
print(f"\nCobertura real: {cubiertas} de {len(cat)} ({100*cubiertas/len(cat):.1f}%)")

# --- 5) las que NO cruzan directo (acá debería salir Danubio) ---
print("\nEstaciones que NO cruzan directo o requirieron normalización:")
flag = cat[cat['estado'] != 'CUBIERTA'].sort_values('estado')
print(flag[['codigo_raw','codigo','nombre_catalogo','estado']].to_string(index=False))

Catálogo oficial: 150 estaciones
Mapeo: 155 códigos -> 148 estaciones únicas

Resumen de cobertura:
estado
CUBIERTA    150

Cobertura real: 150 de 150 (100.0%)

Estaciones que NO cruzan directo o requirieron normalización:
codigo_raw codigo nombre_catalogo                                   estado
      9005  09005         Danubio CUBIERTA (requería zfill: 9005 -> 09005)
